# BirdClef 2026 - Ensemble Inference Notebook

This notebook supports:
- Multiple model architectures (EfficientNet B0/B1/B2/B3)
- PERCH/YAMNet embedding models
- Model ensembles with weighted averaging
- Test-Time Augmentation (TTA)
- GPU/MPS acceleration

## Configuration

Update these paths and settings before running:
- `MODEL_PATHS`: List of model checkpoint paths
- `MODEL_TYPES`: List of backbone types for each model
- `MODEL_WEIGHTS`: Weights for ensemble (optional)
- `USE_TTA`: Enable/disable TTA
- `TTA_AUGMENTS`: TTA augmentation types

## Push to Kaggle
```
kaggle kernels push -p notebooks/
```

In [ ]:
# Install dependencies
!pip install -q torch librosa soundfile pandas numpy torchvision tqdm tf-keras tensorflow tensorflow-hub audioclass

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
import gc

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION - Update these settings for your submission
# ============================================================

# Audio parameters (must match training)
SAMPLE_RATE = 32000
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
DURATION = 5
NUM_CLASSES = 234

# Model configuration - add your model paths here
# Format: [{"path": "/path/to/model.pt", "backbone": "efficientnet_b0", "weight": 1.0, "embedding_model": null}]
# For PERCH/YAMNet: embedding_model can be "perch", "yamnet", or "simple"
MODELS_CONFIG = [
    {
        "path": "/kaggle/input/datasets/krist0phersmith/birdclef2026-model/best_model.pt",
        "backbone": "efficientnet_b2",
        "weight": 1.0,
        "embedding_model": None
    },
    # Add more models here for ensemble:
    # {"path": "/kaggle/input/datasets/krist0phersmith/birdclef2026-model/best_b0.pt", "backbone": "efficientnet_b0", "weight": 0.5, "embedding_model": None},
    # {"path": "/kaggle/input/datasets/krist0phersmith/birdclef2026-model/perch_model.pt", "backbone": "efficientnet_b0", "weight": 0.3, "embedding_model": "perch"},
]

# Ensemble settings
ENSEMBLE_AGGREGATION = "average"  # "average" or "max"

# TTA settings
USE_TTA = False  # Set to True to enable TTA
TTA_AUGMENTS = "original,flip"  # Options: original, flip, timeshift, freqmask, timemask

# Batch size for inference
BATCH_SIZE = 16

print("Configuration loaded!")

## Device Setup

In [ ]:
def get_device():
    """Get the best available device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif device.type == "mps":
    print("Apple Silicon MPS enabled")

## Import Model Definitions

In [ ]:
# Add src to path for imports
sys.path.insert(0, '/kaggle/input/birdclef2026/src')

# Try to import from src, fall back to inline definitions
try:
    from model import BirdClefModel
    from ensemble import EnsemblePredictor
    from tta import get_tta_transforms, apply_tta_to_predictions
    from model_perch import create_embedding_model, PERCH_AVAILABLE as PERCH_AVAIL, YAMNET_AVAILABLE as YAMNET_AVAIL
    print("Successfully imported from src/")
    USE_SRC = True
except ImportError as e:
    print(f"Could not import from src: {e}")
    print("Using inline model definitions")
    USE_SRC = False
    PERCH_AVAIL = False
    YAMNET_AVAIL = False

## Model Definitions (Inline Fallback)

In [ ]:
if not USE_SRC:
    # Inline model definitions if src import fails
    
    class BirdClefModel(nn.Module):
        """EfficientNet-based model for bird species classification."""
        
        def __init__(
            self,
            num_classes: int = 234,
            backbone: str = "efficientnet_b0",
            pretrained: bool = False,
            dropout: float = 0.3,
        ):
            super().__init__()
            
            if backbone == "efficientnet_b0":
                self.backbone = models.efficientnet_b0(weights=None)
                in_features = self.backbone.classifier[1].in_features
            elif backbone == "efficientnet_b1":
                self.backbone = models.efficientnet_b1(weights=None)
                in_features = self.backbone.classifier[1].in_features
            elif backbone == "efficientnet_b2":
                self.backbone = models.efficientnet_b2(weights=None)
                in_features = self.backbone.classifier[1].in_features
            elif backbone == "efficientnet_b3":
                self.backbone = models.efficientnet_b3(weights=None)
                in_features = self.backbone.classifier[1].in_features
            else:
                raise ValueError(f"Unknown backbone: {backbone}")
            
            self.backbone.classifier = nn.Identity()
            self.dropout = nn.Dropout(dropout)
            self.fc = nn.Linear(in_features, num_classes)

        def forward(self, x):
            if x.size(1) == 1:
                x = x.repeat(1, 3, 1, 1)
            x = self.backbone(x)
            x = self.dropout(x)
            x = self.fc(x)
            return x
    
    print("Inline model definitions loaded")

## TTA Definitions (Inline Fallback)

In [ ]:
if not USE_SRC:
    # Inline TTA classes
    
    class TTAAugment:
        def __call__(self, x: torch.Tensor) -> torch.Tensor:
            raise NotImplementedError

    class TTAOriginal(TTAAugment):
        def __call__(self, x):
            return x

    class TTAHorizontalFlip(TTAAugment):
        def __call__(self, x):
            return torch.flip(x, dims=[-1])

    class TTATimeShift(TTAAugment):
        def __init__(self, max_shift: int = 10):
            self.max_shift = max_shift

        def __call__(self, x):
            shift = np.random.randint(-self.max_shift, self.max_shift)
            if shift == 0:
                return x
            if shift > 0:
                return torch.cat([x[:, :, :, shift:], x[:, :, :, :shift]], dim=3)
            else:
                return torch.cat([x[:, :, :, shift:], x[:, :, :, :shift]], dim=3)

    class TTAFreqMask(TTAAugment):
        def __init__(self, freq_mask_param: int = 10, num_masks: int = 1):
            self.freq_mask_param = freq_mask_param
            self.num_masks = num_masks

        def __call__(self, x):
            _, _, freq_bins, _ = x.shape
            for _ in range(self.num_masks):
                f = np.random.randint(0, self.freq_mask_param)
                f0 = np.random.randint(0, max(1, freq_bins - f))
                x[:, :, f0:f0 + f, :] = 0
            return x

    class TTATimeMask(TTAAugment):
        def __init__(self, time_mask_param: int = 20, num_masks: int = 1):
            self.time_mask_param = time_mask_param
            self.num_masks = num_masks

        def __call__(self, x):
            _, _, _, time_bins = x.shape
            for _ in range(self.num_masks):
                t = np.random.randint(0, self.time_mask_param)
                t0 = np.random.randint(0, max(1, time_bins - t))
                x[:, :, :, t0:t0 + t] = 0
            return x

    def get_tta_transforms(augments: str = "original,flip"):
        """Get TTA transforms based on string specification."""
        transforms = []
        augment_list = [a.strip() for a in augments.split(',')]
        
        for aug in augment_list:
            if aug == "original":
                transforms.append(TTAOriginal())
            elif aug == "flip":
                transforms.append(TTAHorizontalFlip())
            elif aug == "timeshift":
                transforms.append(TTATimeShift(max_shift=10))
            elif aug == "freqmask":
                transforms.append(TTAFreqMask(freq_mask_param=10))
            elif aug == "timemask":
                transforms.append(TTATimeMask(time_mask_param=20))
        
        return transforms
    
    def apply_tta_to_predictions(model, inputs, augments, device):
        """Apply TTA and average predictions."""
        all_preds = []
        
        for aug in augments:
            augmented = aug(inputs)
            augmented = augmented.to(device)
            with torch.no_grad():
                logits = model(augmented)
                probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu())
        
        return torch.stack(all_preds).mean(dim=0)
    
    print("Inline TTA definitions loaded")

## Load Models

In [ ]:
# Check PERCH availability
if USE_SRC:
    print(f"PERCH available: {PERCH_AVAIL}")
    print(f"YAMNet available: {YAMNET_AVAIL}")
else:
    print("PERCH/YAMNet: Not available (src not imported)")

In [ ]:
# Load models based on configuration
models = []
model_info = []

for i, config in enumerate(MODELS_CONFIG):
    model_path = config["path"]
    backbone = config.get("backbone", "efficientnet_b0")
    embedding_model = config.get("embedding_model", None)
    weight = config.get("weight", 1.0)
    
    print(f"\nLoading model {i+1}/{len(MODELS_CONFIG)}: {os.path.basename(model_path)}")
    print(f"  Backbone: {backbone}")
    print(f"  Embedding: {embedding_model}")
    print(f"  Weight: {weight}")
    
    if not os.path.exists(model_path):
        print(f"  WARNING: Model file not found at {model_path}")
        continue
    
    try:
        # Check if using embedding model (PERCH/YAMNet)
        if embedding_model and USE_SRC:
            if embedding_model == "perch" and not PERCH_AVAIL:
                print(f"  WARNING: PERCH not available, skipping")
                continue
            if embedding_model == "yamnet" and not YAMNET_AVAIL:
                print(f"  WARNING: YAMNet not available, skipping")
                continue
            
            model = create_embedding_model(
                model_type=embedding_model,
                num_classes=NUM_CLASSES,
                pretrained=False,
            )
        else:
            model = BirdClefModel(
                num_classes=NUM_CLASSES,
                backbone=backbone,
                pretrained=False,
            )
        
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        
        # Try to load state dict
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'], strict=False)
            print(f"  Loaded from checkpoint - val_loss: {checkpoint.get('val_loss', 'N/A')}")
        else:
            model.load_state_dict(checkpoint, strict=False)
            print(f"  Loaded state dict directly")
        
        model = model.to(device)
        model.eval()
        
        models.append(model)
        model_info.append({"weight": weight, "backbone": backbone, "embedding": embedding_model})
        
        print(f"  Successfully loaded!")
        
    except Exception as e:
        print(f"  ERROR loading model: {e}")
        continue

print(f"\n{'='*50}")
print(f"Loaded {len(models)} models for ensemble")

## TTA Setup

In [ ]:
# Setup TTA transforms
if USE_TTA and len(models) > 0:
    tta_transforms = get_tta_transforms(TTA_AUGMENTS)
    print(f"TTA enabled with {len(tta_transforms)} transforms: {TTA_AUGMENTS}")
else:
    tta_transforms = None
    if not USE_TTA:
        print("TTA disabled")
    else:
        print("No models loaded, skipping TTA")

## Load Taxonomy and Submission Format

In [ ]:
# Load taxonomy to get class names
taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
label_cols = taxonomy['primary_label'].tolist()
print(f"Number of classes: {len(label_cols)}")

# Load sample submission
sample_sub = pd.read_csv('/kaggle/input/competitions/birdclef-2026/sample_submission.csv')
submission_label_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f"Submission columns: {len(submission_label_cols)}")
print(f"Sample submission shape: {sample_sub.shape}")

## Audio Processing Functions

In [ ]:
def compute_spectrogram(y, sr):
    """Compute normalized mel spectrogram."""
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)

def load_and_process_audio(audio_path):
    """Process audio file into 5-second segments."""
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    
    segment_samples = SAMPLE_RATE * DURATION
    spectrograms = []
    row_ids = []
    
    for start in range(0, len(y), segment_samples):
        end = min(start + segment_samples, len(y))
        segment = y[start:end]
        
        if len(segment) < segment_samples:
            segment = np.pad(segment, (0, segment_samples - len(segment)))
        
        spec = compute_spectrogram(segment, sr)
        spectrograms.append(spec)
        
        filename = os.path.basename(audio_path).replace('.ogg', '')
        start_sec = start // SAMPLE_RATE
        row_id = f"{filename}_{start_sec + DURATION}"
        row_ids.append(row_id)
    
    return spectrograms, row_ids

## Get Test Files

In [ ]:
# Find test audio files
test_dir = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'

if os.path.exists(test_dir):
    test_files = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.ogg')]
    print(f"Found {len(test_files)} test files")
else:
    test_files = []
    print("Test directory not found — no test files available (expected during non-submission runs).")

## Run Inference

In [ ]:
if len(models) == 0:
    print("ERROR: No models loaded, cannot run inference")
else:
    # Extract weights for ensemble
    weights = [info["weight"] for info in model_info]
    total_weight = sum(weights)
    if total_weight > 0:
        weights = [w / total_weight for w in weights]  # Normalize
    
    print(f"Ensemble weights: {weights}")
    print(f"Aggregation: {ENSEMBLE_AGGREGATION}")
    
    all_predictions = []
    all_row_ids = []

    for audio_path in tqdm(test_files, desc="Processing audio"):
        spectrograms, row_ids = load_and_process_audio(audio_path)
        
        if not spectrograms:
            continue
        
        # Convert to tensor
        specs_tensor = torch.tensor(np.array(spectrograms)).unsqueeze(1)
        
        # Make predictions with ensemble
        predictions = []
        
        for i in range(0, len(specs_tensor), BATCH_SIZE):
            batch = specs_tensor[i:i+BATCH_SIZE]
            batch_preds = []
            
            for model, weight in zip(models, weights):
                batch = batch.to(device)
                
                if USE_TTA and tta_transforms:
                    # Apply TTA
                    probs = apply_tta_to_predictions(model, batch, tta_transforms, device)
                else:
                    with torch.no_grad():
                        outputs = model(batch)
                        probs = torch.sigmoid(outputs)
                
                batch_preds.append(probs.cpu() * weight)
            
            # Aggregate across models
            if ENSEMBLE_AGGREGATION == "average":
                final_preds = torch.stack(batch_preds).sum(dim=0)
            else:  # max
                final_preds = torch.stack(batch_preds).max(dim=0)[0]
            
            predictions.append(final_preds.numpy())
        
        predictions = np.concatenate(predictions, axis=0)
        all_predictions.append(predictions)
        all_row_ids.extend(row_ids)

    print(f"Total segments: {len(all_row_ids)}")

## Create Submission

In [ ]:
if all_predictions:
    print(f"Building submission from {len(all_row_ids)} predicted segments...")
    predictions_array = np.concatenate(all_predictions, axis=0)

    submission = pd.DataFrame({'row_id': all_row_ids})
    label_to_idx = {label: i for i, label in enumerate(label_cols)}
    
    # Map predictions to submission columns
    for col in submission_label_cols:
        if col in label_to_idx:
            model_idx = label_to_idx[col]
            submission[col] = predictions_array[:, model_idx]
        else:
            print(f"Warning: {col} not in training labels, using default")
            submission[col] = 1.0 / NUM_CLASSES

    # Left-join onto sample_sub so row order and any missing rows are handled correctly
    final_submission = sample_sub[['row_id']].merge(submission, on='row_id', how='left')
    final_submission[submission_label_cols] = final_submission[submission_label_cols].fillna(1 / NUM_CLASSES)
    print(f"Submission shape: {final_submission.shape}")

else:
    # No test files found
    print("No test files found — using sample submission defaults (non-submission run).")
    final_submission = sample_sub.copy()
    print(f"Submission shape: {final_submission.shape}")

## Validation Checks

In [ ]:
# Sanity check
expected_row_ids = set(sample_sub['row_id'].tolist())
output_row_ids = set(final_submission['row_id'].tolist())

missing = expected_row_ids - output_row_ids
extra = output_row_ids - expected_row_ids
nan_count = final_submission[submission_label_cols].isna().sum().sum()

print(f"Expected row_ids : {len(expected_row_ids)}")
print(f"Output row_ids   : {len(output_row_ids)}")
print(f"Missing row_ids  : {len(missing)}")
print(f"Extra row_ids    : {len(extra)}")
print(f"NaN values       : {nan_count}")

if not missing and not extra and nan_count == 0:
    print("\n✓ All checks passed!")
else:
    print("\n⚠ WARNING: issues found above.")

## Save Submission

In [ ]:
# Save submission
final_submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

# Verify
sub_check = pd.read_csv('submission.csv')
print(f"Shape: {sub_check.shape}")
print(sub_check.head())